# Fəsil 18 — Kompüter görməsi və təsvirlərin emalı

Bu notebook MDU magistr tələbələri üçün nəzəriyyə və praktikanı birləşdirir. Hüceyrələri ardıcıllıqla icra edin.

## 1. Mühitin hazırlanması
Google Colab-da GPU mövcudluğu hesab və istifadə limitlərindən asılıdır. Runtime → Change runtime type bölməsindən GPU seçilə bilər.

## Real nümunə: Ultralytics avtobus fotosu
Aşağıdakı hüceyrə sənədlərdə işlədilən real demo görüntüsünü yükləyir və YOLO ilə aşkarlama aparır. İnternet tələb olunur.

In [ ]:
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import requests
image_url = 'https://ultralytics.com/images/bus.jpg'
r = requests.get(image_url, timeout=30); r.raise_for_status()
open('bus.jpg','wb').write(r.content)
model = YOLO('yolo26n.pt')
result = model.predict(source='bus.jpg', conf=0.25)[0]
annotated = result.plot()
display(Image.fromarray(annotated[:,:,::-1]))
result.save(filename='bus_detection_result.jpg')


In [ ]:
!pip -q install opencv-python ultralytics matplotlib

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt, torch, os
print('OpenCV:', cv2.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 2. Şəkil yükləmə və ilkin yoxlama

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = next(iter(uploaded.keys()))
img = cv2.imread(image_path)
if img is None: raise ValueError('Şəkil oxunmadı')
print('shape:', img.shape, 'dtype:', img.dtype, 'min/max:', img.min(), img.max())
plt.figure(figsize=(8,5)); plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.axis('off');

## 3. Rəng modelləri

In [ ]:
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
fig, ax = plt.subplots(1,3,figsize=(15,4))
ax[0].imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB)); ax[0].set_title('RGB')
ax[1].imshow(gray,cmap='gray'); ax[1].set_title('Grayscale')
ax[2].imshow(hsv); ax[2].set_title('HSV (vizualizasiya üçün RGB kimi deyil)')
for a in ax:a.axis('off')

## 4. Həndəsi çevrilmələr

In [ ]:
h,w = img.shape[:2]
small = cv2.resize(img,(int(w*.5),int(h*.5)),interpolation=cv2.INTER_AREA)
M = cv2.getRotationMatrix2D((w/2,h/2),25,1.0)
rot = cv2.warpAffine(img,M,(w,h))
crop = img[h//4:3*h//4,w//4:3*w//4]
fig,ax=plt.subplots(1,3,figsize=(14,4))
for a,x,t in zip(ax,[small,rot,crop],['50% ölçü','25° döndərmə','Mərkəzi crop']):a.imshow(cv2.cvtColor(x,cv2.COLOR_BGR2RGB));a.set_title(t);a.axis('off')

## 5. Filtrlər və kənarların aşkarlanması

In [ ]:
gauss=cv2.GaussianBlur(img,(5,5),0)
median=cv2.medianBlur(img,5)
bilateral=cv2.bilateralFilter(img,9,75,75)
edges=cv2.Canny(cv2.cvtColor(gauss,cv2.COLOR_BGR2GRAY),80,160)
fig,ax=plt.subplots(1,4,figsize=(16,4))
for a,x,t in zip(ax,[gauss,median,bilateral,edges],['Gaussian','Median','Bilateral','Canny']):
    a.imshow(x if x.ndim==2 else cv2.cvtColor(x,cv2.COLOR_BGR2RGB),cmap='gray' if x.ndim==2 else None);a.set_title(t);a.axis('off')

## 6. Thresholding, morfologiya və konturlar

In [ ]:
_,th=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU)
k=cv2.getStructuringElement(cv2.MORPH_RECT,(3,3))
opened=cv2.morphologyEx(th,cv2.MORPH_OPEN,k)
contours,_=cv2.findContours(opened,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
out=img.copy()
cv2.drawContours(out,[c for c in contours if cv2.contourArea(c)>100],-1,(0,255,0),2)
print('Konturlar:',len(contours))
plt.figure(figsize=(12,4));plt.subplot(1,3,1);plt.imshow(th,cmap='gray');plt.title('Otsu');plt.subplot(1,3,2);plt.imshow(opened,cmap='gray');plt.title('Opening');plt.subplot(1,3,3);plt.imshow(cv2.cvtColor(out,cv2.COLOR_BGR2RGB));plt.title('Konturlar');

## 7. Histogram və statistika

In [ ]:
mean,std=cv2.meanStdDev(gray)
print('Orta intensivlik:',float(mean[0][0]),'Standart kənarlaşma:',float(std[0][0]))
plt.figure(figsize=(8,3));plt.hist(gray.ravel(),bins=256,range=(0,256));plt.title('Grayscale histogram');plt.xlabel('İntensivlik');plt.ylabel('Piksel sayı');

## 8. Video faylını yükləyin və YOLO ilə aşkarlama edin
Qısa, istifadə hüququ olan MP4 faylı yükləyin. Colab mühitində lokal USB kameraya birbaşa `source=0` çıxışı adətən olmur.

In [ ]:
video_upload=files.upload()
video_path=next(iter(video_upload.keys()))
cap=cv2.VideoCapture(video_path)
print('Açıldı:',cap.isOpened(),'FPS:',cap.get(cv2.CAP_PROP_FPS),'Frames:',cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

In [ ]:
from ultralytics import YOLO
model=YOLO('yolo26n.pt')
results=model.predict(source=video_path,conf=0.25,save=True,stream=True)
for idx,result in enumerate(results):
    print('Frame',idx,'detection count:',len(result.boxes))
    if idx>=9: break  # ilk 10 kadrı nümunə üçün emal et

## 9. Nəticənin şərhi və hesabat
- Modelin tanıdığı sinifləri sadalayın.
- Confidence həddini 0.20, 0.35, 0.50 dəyişərək nəticəni müqayisə edin.
- Ən azı 3 yanlış müsbət və ya yanlış mənfi halı təhlil edin.
- Video kadrlarını train/test-ə təsadüfi bölməyin; eyni çəkilişdən kadrlar məlumat sızmasına səbəb ola bilər.
- Dataset/video lisenziyasını və məxfilik məsələlərini qeyd edin.